# Phase 1: ResNet-50 on Colab, all holdouts in one run
1. Runtime → Change runtime type → **T4 GPU**
2. Runtime → **Run all**, then approve the Google Drive sign-in popup (cell 2). The run can't start until you do.
3. Leave it running. Each holdout (1–5) is saved to `MyDrive/dl_results/` as soon as it finishes. If Colab disconnects, click **Run all** again: finished holdouts are skipped.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RESULTS = '/content/drive/MyDrive/dl_results'

In [ ]:
import os
if not os.path.exists('/content/repo'):
    !git clone https://github.com/BraydenHuguenard/Generalization-of-AI-Generated-Image-Detectors-to-Unseen-Generators.git /content/repo
else:
    !git -C /content/repo pull
%cd /content/repo
!pip install -q datasets scikit-learn
# download the dataset (~7.5 GB) once up front so every run reuses the local cache
!python -c "from datasets import load_dataset; print(load_dataset('Rajarshi-Roy-research/Defactify_Image_Dataset'))"

Settings. These defaults are sized so all five holdouts finish in roughly an hour on a T4, including the download.
Check the time printed after the first holdout: if it's much faster than ~8 min, raise `MAX_TRAIN`/`EPOCHS`; if it's slower, lower them.
`MAX_TRAIN = None` uses the full training set, which takes much longer.

In [ ]:
HOLDOUTS = [1, 2, 3, 4, 5]   # one run per generator held out
EPOCHS = 3
MAX_TRAIN = 12000            # training images per run (None = all)
MAX_EVAL = 2000              # val/test images per run (None = all)
BATCH_SIZE = 64

Train every holdout, then make that run's charts. Holdouts that already have `metrics.json` in Drive are skipped.

In [ ]:
import time
from IPython.display import Image, display

limits = (f"--max-train {MAX_TRAIN} " if MAX_TRAIN else "") + (f"--max-eval {MAX_EVAL}" if MAX_EVAL else "")
start = time.time()
for h in HOLDOUTS:
    out = f"{RESULTS}/holdout_{h}"
    if os.path.exists(f"{out}/metrics.json"):
        print(f"holdout {h}: already done, skipping")
        continue
    t0 = time.time()
    print(f"\n========== holdout {h} ==========")
    !python main.py --holdout {h} --epochs {EPOCHS} --batch-size {BATCH_SIZE} --workers 2 {limits} --out {out}
    !python scripts/plot_training.py {out}
    !python scripts/plot_generators.py {out}
    print(f"holdout {h} took {(time.time() - t0) / 60:.1f} min (total {(time.time() - start) / 60:.1f} min)")

Compare all holdouts (seen vs unseen and the generalization gap).

In [ ]:
done = [f"{RESULTS}/holdout_{h}" for h in HOLDOUTS if os.path.exists(f"{RESULTS}/holdout_{h}/metrics.json")]
!python scripts/plot_compare.py {' '.join(done)} --out {RESULTS}
display(Image(f'{RESULTS}/comparison_roc_auc.png'))

Per-holdout charts (also saved as PNGs in each Drive folder).

In [ ]:
for d in done:
    print(d)
    for f in ['training_curves.png', 'per_generator_roc_auc.png', 'roc_curves.png', 'confusion_matrices.png']:
        if os.path.exists(f'{d}/{f}'):
            display(Image(f'{d}/{f}'))